In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca": "../../data/reduced/dataPCA.csv",
    "fs": "../../data/reduced/dataFS.csv"
}

outDir = "../../results/hdbscan"

In [12]:
def plotHdbscan(df, labels, datasetName):
    pca2d = PCA(n_components=2, random_state=42)
    X2d = pca2d.fit_transform(df)

    plt.figure(figsize=(6, 5))

    noiseMask = labels == -1
    plt.scatter(
        X2d[noiseMask, 0],
        X2d[noiseMask, 1],
        c="lightgray",
        s=5,
        label="Noise"
    )

    clusterMask = labels != -1
    plt.scatter(
        X2d[clusterMask, 0],
        X2d[clusterMask, 1],
        c=labels[clusterMask],
        cmap="tab10",
        s=5,
        label="Clusters"
    )

    plt.title(f"HDBSCAN ({datasetName})")
    plt.legend(markerscale=2)
    plt.tight_layout()
    plt.savefig(f"{outDir}/hdbscan_{datasetName}.png", dpi=200)
    plt.close()

In [13]:
minClusterSize = 15
minSamples = 10

def runHdbscan(datasetName, df):

    model = hdbscan.HDBSCAN(
        min_cluster_size=minClusterSize,
        min_samples=minSamples,
        metric="euclidean"
    )

    labels = model.fit_predict(df)

    numClusters = len(set(labels) - {-1})
    noiseRatio = np.sum(labels == -1) / len(labels)

    coreMask = labels != -1
    uniqueClusters = set(labels[coreMask])

    
    sil = silhouette_score(df[coreMask], labels[coreMask])
    db = davies_bouldin_score(df[coreMask], labels[coreMask])

    resultData = df.copy()
    resultData["cluster"] = labels
    resultData.to_csv(
        f"{outDir}/hdbscan_{datasetName}_clusters.csv",
        index=False
    )

    plotHdbscan(df, labels, datasetName)

    return {
        "Dataset": datasetName,
        "Algorithm": "HDBSCAN",
        "min_cluster_size": minClusterSize,
        "min_samples": minSamples,
        "clusters": numClusters,
        "noise_ratio": noiseRatio,
        "Silhouette": sil,
        "Davies_Bouldin": db
    }

In [14]:
results = []

for name, path in datasets.items():
    df = pd.read_csv(path)
    if "Class" in df.columns:
        y = df["Class"].values
        X = df.drop(columns=["Class"])
    else: 
        X = df
    summary = runHdbscan(name, X)
    results.append(summary)

In [15]:
summaryData = pd.DataFrame(results)
summaryData.to_csv(f"{outDir}/hdbscan_summary.csv", index=False)
print(summaryData)

  Dataset Algorithm  min_cluster_size  min_samples  clusters  noise_ratio  \
0    full   HDBSCAN                15           10       213     0.319698   
1     pca   HDBSCAN                15           10       223     0.306183   
2      fs   HDBSCAN                15           10       186     0.395327   

   Silhouette  Davies_Bouldin  
0    0.430574        0.937362  
1    0.482527        0.924494  
2    0.462349        0.778126  
